# LoKI reduction on `local()`

The specs and bindings of `ess.instruments.loki` on the runs of the LoKI detector test at Larmor, without a GUI.
A reduction is three records, and a person can look at each before the next:

```
mask files, one run ──> masks ──> beam centre ──> I(Q)
                          └───────────────────────^
```

## A folder of files, and a table

`FolderFiles` serves the files of one folder, with metadata from a table a person fills.
Here the folder is the download cache of esssans' tutorial data, and the first cell downloads the files that are missing.

Run 60392 is the can's transmission run and the empty beam. A file has one role, so it has the role `transmission`, and the spec types `empty_beam` as a run of any role (#153).

In [ ]:
from pathlib import Path

import scipp as sc
from ess.loki import data

from ess.apps import FolderFiles, Template, file, local
from ess.instruments.loki.bindings import BINDINGS
from ess.instruments.loki.specs import BEAM_CENTRE, IOFQ, MASKS

for download in (
    data.loki_tutorial_run_60392,
    data.loki_tutorial_background_run_60393,
    data.loki_tutorial_sample_run_60339,
    data.loki_tutorial_isis_polymer_sample_run,
    data.loki_tutorial_isis_polymer_transmission_run,
    data.loki_tutorial_agbeh_sample_run,
    data.loki_tutorial_agbeh_transmission_run,
    data.loki_tutorial_porous_silica_sample_run,
    data.loki_tutorial_porous_silica_transmission_run,
    data.loki_tutorial_deut_sds_sample_run,
    data.loki_tutorial_deut_sds_transmission_run,
    data.loki_tutorial_direct_beam_all_pixels,
):
    download()
mask_file = Path(data.loki_tutorial_mask_filenames()[0])
direct_beam_file = Path(data.loki_tutorial_direct_beam_all_pixels())

In [ ]:
def run(number: int, role: str, sample: str | None = None) -> dict[str, object]:
    row = {'file': f'{number}-2022-02-28_2215.nxs', 'run': number, 'role': role}
    return row | ({'sample': sample} if sample else {})


table = [
    run(60339, 'sample'),
    run(60395, 'sample', 'ISIS polymer'),
    run(60394, 'transmission', 'ISIS polymer'),
    run(60387, 'sample', 'AgBeh'),
    run(60386, 'transmission', 'AgBeh'),
    run(60385, 'sample', 'porous silica'),
    run(60384, 'transmission', 'porous silica'),
    run(60389, 'sample', 'deut-SDS'),
    run(60388, 'transmission', 'deut-SDS'),
    run(60393, 'background'),
    run(60392, 'transmission'),
    {'file': mask_file.name, 'kind': 'mask'},
    {'file': direct_beam_file.name, 'kind': 'calibration', 'role': 'direct_beam'},
]
files = FolderFiles(mask_file.parent, table)
client = local(proposal='loki-test', files=files, bind=BINDINGS)

## Masks

`MASKS` reads the mask files and shows them on the detector image of a run.
The image holds the counts per pixel, with each pixel's position and the masks, so a plot can leave the masked pixels out or show what they hold.

In [ ]:
masks = client.submit(
    MASKS, params={'run': file(run=60339), 'mask_files': [file(path=mask_file)]}
)
client.wait(masks)
image = client.output(masks, 'image').counts
image

In [ ]:
from ess.instruments.loki.plots import on_detector

sc.plot(on_detector(image, masked=False), norm='log', title='All pixels')

In [ ]:
sc.plot(on_detector(image), norm='log', title='Without the masked pixels')

## Beam centre

`BEAM_CENTRE` finds the centre of mass of the counts outside the masks: a point on the beam relative to the sample, in the same frame as the image's positions, with `z` the distance from the sample.
The plot marks the centre on the image, from two outputs of one record.

In [ ]:
centre = client.submit(
    BEAM_CENTRE, params={'sample': file(run=60339), 'masks': masks.ref('masks')}
)
client.wait(centre)
found = client.output(centre, 'centre')
found

In [ ]:
figure = sc.plot(
    on_detector(client.output(centre, 'image').counts),
    norm='log',
    title=f'Beam centre: x = {found.x:.4f} m, y = {found.y:.4f} m',
)
figure.ax.plot(found.x, found.y, marker='+', color='red', markersize=20, mew=2)
figure

## I(Q)

`IOFQ` takes the masks and the centre as references to the records above.

In [ ]:
settings = {
    'can': file(run=60393),
    'can_transmission': file(run=60392),
    'empty_beam': file(run=60392),
    'direct_beam': file(path=direct_beam_file),
    'masks': masks.ref('masks'),
    'beam_centre': centre.ref('centre'),
    'wavelength': {'start': 1.0, 'stop': 13.0, 'num_bins': 200},
    'q': {'start': 0.01, 'stop': 0.3, 'num_bins': 100},
}
iofq = client.compute(
    IOFQ,
    params={
        **settings,
        'sample': file(run=60339),
        'sample_transmission': file(run=60394),
    },
)
result = client.output(iofq)
sc.plot(result['iofq'].intensity, norm='log', title='I(Q) of run 60339')

In [ ]:
curves = result['transmission']
sc.plot({'sample': curves.sample, 'can': curves.can}, title='Transmission')

## A batch with one can

Four samples, each with its transmission run, against one can.
The masks are an input of `IOFQ`, so the can's branch does not depend on the sample, and the stage reduces the can once.

In [ ]:
samples = {
    'ISIS polymer': (60395, 60394),
    'AgBeh': (60387, 60386),
    'porous silica': (60385, 60384),
    'deut-SDS': (60389, 60388),
}
batch = client.stage(
    Template(IOFQ, params=settings, blanks=('sample', 'sample_transmission'))
)
records = {
    name: client.submit(
        batch,
        params={'sample': file(run=run), 'sample_transmission': file(run=trans)},
        label='iofq',
        member=name,
    )
    for name, (run, trans) in samples.items()
}
client.wait(list(records.values()))
sc.plot(
    {name: client.output(r, 'iofq').intensity for name, r in records.items()},
    norm='log',
    title='I(Q) per sample',
)

## Tweaking one run

A stage with `q` blank holds everything up to the binning in Q, so another binning does not load the runs again.

In [ ]:
tune = client.stage(
    Template(
        IOFQ,
        params={
            **settings,
            'sample': file(run=60387),
            'sample_transmission': file(run=60386),
        },
        blanks=('q',),
    )
)
tuned = {
    bins: client.submit(tune, params={'q': {'start': 0.01, 'stop': 0.3, 'num_bins': bins}})
    for bins in (50, 100, 200)
}
client.wait(list(tuned.values()))
sc.plot(
    {f'{b} bins': client.output(r, 'iofq').intensity for b, r in tuned.items()},
    norm='log',
    title='AgBeh',
)

## What a result came from

Each record names the records and files it came from.

In [ ]:
provenance = client.provenance(records['AgBeh'])
[record.spec.name for record in provenance.records()], [str(f) for f in provenance.files()]

In [ ]:
client.close()